# Local multi-turn SQL lab

This is the shareable notebook attached to the blog post's codebase. It runs a compact in-memory SQLite lab that asks one question: which training target actually helps a local model handle multi-turn data analysis?

The notebook does not download a model or start a serving stack. It uses `auto` runtime selection: CUDA, MPS, or XPU when PyTorch can see one, otherwise CPU.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "notebooks" / "labs").exists():
        repo_root = candidate
        break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
import pandas as pd

from notebooks.labs.local_multiturn_sql_lab_support import (
    lab_turns,
    run_multiturn_lab,
)


## 1. Pick a portable runtime

Leave `DEVICE = "auto"` for the public lab. It will use CUDA, MPS, or XPU if available and will default to CPU otherwise. You can force `"cpu"`, `"cuda"`, `"mps"`, or `"xpu"` when checking a specific machine.

In [ ]:
DEVICE = "auto"
report = run_multiturn_lab(device_preference=DEVICE)

assert report["device"].kind in {"cpu", "cuda", "mps", "xpu"}
assert report["runtime_policy"]["accelerator_usage"] == "selected_if_available"

runtime_summary = {
    "requested_device": DEVICE,
    "selected_device": report["device"].kind,
    "selected_label": report["device"].label,
    "detected_accelerator": report["detected_accelerator"].label,
    "fallback": report["runtime_policy"]["fallback"] or "none",
    "scenario_hash": report["scenario_contract"]["shared_input_sha256"],
}
runtime_summary


In [ ]:
pd.DataFrame(report["accelerator_report"])


## 2. Inspect the multi-turn task

The toy warehouse has customers and orders. The four turns intentionally stress different pieces of conversational analysis: keep the metric, apply a follow-up filter, change the grain, and recover after an empty result.

In [ ]:
turns = pd.DataFrame(
    [
        {
            "turn_id": turn.turn_id,
            "question": turn.question,
            "context_note": turn.context_note,
            "requires_recovery": turn.requires_recovery,
            "reference_sql": turn.reference_sql,
        }
        for turn in lab_turns()
    ]
)
turns


## 3. Compare candidate training targets

The lab compares direct SQL with four richer targets: planner-first SQL, semantic value grounding, a `MEASURE()`-preserving DSL, and behavior/recovery tuning.

In [ ]:
pd.DataFrame(report["method_matrix"])


## 4. Run the lab scorecard

`value_accuracy` is the execution result match. The other columns expose why a system got there: context carryover, value grounding, metric preservation, and recovery.

In [ ]:
scores = pd.DataFrame(
    [
        {"system": system, **metrics}
        for system, metrics in report["systems"].items()
    ]
).sort_values("value_accuracy", ascending=False)
scores


In [ ]:
assert scores.loc[scores["system"] == "direct_sql_baseline", "value_accuracy"].item() == 0.25
assert scores.loc[scores["system"] == "semantic_dsl_planner", "measure_preservation_rate"].item() == 1.0
assert scores.loc[scores["system"] == "behavior_recovery_sql", "recovery_success_rate"].item() == 1.0


## 5. Read the trace, not only the score

A final SQL answer can be wrong for several different reasons. The trace keeps those reasons separate so the next dataset or fine-tuning run has a concrete target.

In [ ]:
trace_columns = [
    "turn_id",
    "question",
    "system",
    "value_match",
    "context_carryover",
    "value_grounded",
    "measure_preserved",
    "recovery_success",
    "failure_type",
]
trace = pd.DataFrame(report["rows"])
trace[trace_columns]


In [ ]:
failures = trace.loc[~trace["value_match"], trace_columns + ["intermediate_plan", "sql", "actual_rows", "expected_rows"]]
failures


## 6. Inspect the intermediate state

This is the reason the project moved beyond plain SQL strings. The better targets make state explicit before SQL execution: the planner carries filters and grain, semantic grounding maps display values to storage values, the DSL preserves governed metrics, and recovery uses feedback from the previous turn.

In [ ]:
plans = trace[["turn_id", "system", "intermediate_plan", "sql"]]
plans


## 7. What this lab proves

This notebook is not a benchmark result and it does not compare against hosted SOTA. It is a small executable lab for deciding what the codebase should evaluate next. The repo evidence should promote a target only after it beats direct SQL on the same rows, scorer, oracle policy, and endpoint path.

In [ ]:
next_gates = pd.DataFrame(
    [
        {
            "target": "planner-first SQL",
            "next_gate": "predict non-oracle plans, then compare generated SQL against same-model direct SQL",
        },
        {
            "target": "semantic value grounding",
            "next_gate": "build value/entity indexes and score value normalization separately",
        },
        {
            "target": "MEASURE()-preserving DSL",
            "next_gate": "compare DSL-preserving runs against direct SQL on identical rows",
        },
        {
            "target": "behavior/recovery tuning",
            "next_gate": "evaluate generated-history rollouts, not only teacher-forced turns",
        },
    ]
)
next_gates
